# 얘! clinvar도 EDA가 된단다!
- 저게 vcf파일이라 csv로 변환은 한번 했습니다... 변환하고 확인하는데 1분 걸리더군요. 저장 30초 로딩 30초...
- 근데 얘가 업데이트가 되잖아요? 그죠 나중에 새로운 파일 나오면 또 저장해야죠. 근데 주기가 그렇게 길진 않습니다. 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.cm as cm

In [ ]:
# 그래프 기본 테마 설정
sns.set_theme(palette="Purples_r", style="whitegrid", font_scale=1) # 블루톤

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Nanumsquare_ac' # 제가... 픽셀체 이런거 좋아해서...
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['axes.titlesize'] = 16 # 제목 폰트 사이즈
plt.rcParams['axes.labelsize'] = 14 # 라벨 폰트 사이즈
plt.rcParams['font.size'] = 14 # 기본 폰트사이즈
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
clinvar_df = pd.read_csv('data/clinvar_20260208.csv', low_memory=False)

# 데이터프레임 확인

## .shape

In [ ]:
clinvar_df.shape

## .info()

In [ ]:
clinvar_df.info()

## .describe()

In [ ]:
clinvar_df.describe(include='all')

## .isna().sum()

In [ ]:
clinvar_df.isna().sum()

- 근데 저거 다 때워야되나? 안 쓰는 칼럼들은 버려도 될 것 같은데.

## .head()

In [ ]:
clinvar_df.head()

## .columns()

In [ ]:
clinvar_df.columns

- 일단 쓸 칼럼 추리는게 일이겠구만...

# 전처리

## 쓸 칼럼만 추리기
- 저거 많은데 분석할 때 쓸만한게 몇개 안됩니다. 예.

### 추릴 칼럼 리스트
1. CHROM: 염색체(몇 번 염색체인지)
2. POS: 염색체 어디?
3. REF, ALT: 비포&애프터 (REF에 있는 시퀀스가 ALT로 바뀐 변이다)
4. CLNSIG: 임상적 유의성
5. CLNVC: 변이 타입(얘가 껴들어간겨 빠진겨 바뀐겨 뒤집어진겨)
6. GENEINFO: 유전자 이름+Entrez ID
7. CLNREVSTAT: 별점?

In [ ]:
analysis_column = ['CHROM','POS','REF','ALT','CLNSIG','CLNVC','GENEINFO','CLNREVSTAT'] # 칼럼 뭐하는건지 위에 있어요
clinvar_df_analysis = clinvar_df[analysis_column]

clinvar_df_analysis

## 결측값 재확인

In [ ]:
clinvar_df_analysis.isna().sum()
# 그러고도 결측값이 있으면 어쩌라는겁니까

In [ ]:
# 어느놈이 비었는지 한번 보겠습니다.
clinvar_df_analysis.query('CLNSIG.isna()').index

In [ ]:
clinvar_df_analysis.loc[17]

- 이거 원본 확인해보니까 진짜로 데이터가 없었습니다. 걍 언노운 처리 하면 될듯?

In [ ]:
# 어느놈이 비었는지 한번 보겠습니다.
clinvar_df_analysis.query('GENEINFO.isna()').index

In [ ]:
clinvar_df_analysis.loc[1927]

- 이쪽도 진짜 데이터가 없는거라서 걍 때우면 될듯.

### 결측값 땜질_최종.py

In [ ]:
fill_values = {
    'CLNSIG': 'Unknown_Significance',
    'CLNREVSTAT': 'No_Assertion',
    'GENEINFO': 'Unknown_Gene',
    'MC': 'Unknown_Consequence'
}

clinvar_df_analysis = clinvar_df_analysis.fillna(value=fill_values)

clinvar_df_analysis['GENE_SYMBOL'] = clinvar_df_analysis['GENEINFO'].apply(
    lambda x: x.split(':')[0] if ':' in x else x
)

In [ ]:
clinvar_df_analysis

## 돌연변이 뭐 있나?

In [ ]:
clinvar_df_analysis['CLNVC'].value_counts()

1. single nucleotide variant: 염기 하나가 다른 변이입니다. 대표적인 예시는 EGFR L858R. L이랑 R은 아미노산인데, 염기 하나가 바뀌어서 코돈이 변경됨->아미노산이 바뀌는겁니다. 나비효과이기도 하고 제일 잡기 어렵습니다.
2. Deletion: 염색체나 염기서열 일부가 있었는데요. 없었습니다. 있었어요. 근데 사라졌어요. ~~노쇼~~
3. Duplication: 편의점 1+1도 아니고 염색체 일부 구간이 복제되는 변이입니다.
4. Microsatellite: 게놈 전반에 걸쳐 2~7bp의 짧은 염기 서열이 반복되는 부위의 변이입니다. 음... 반복작업 하다 뻑나는건가... (처음 보는 변이임)
5. Indel: DNA를 복제할 때 염기가 첨삭되는 변이입니다.
6. Insertion: 염색체나 염기서열 일부가 끼어드는 변이입니다. ~~새치기~~
7. Inversion: 염기서열 일부가 **반대로** 들어갔습니다. DNA도 방향이 있어요 여러분...
8. Variation: 같은 종 또는 번식 집단 내 개체들 사이에서 형태, 생리, 행동 등 형질이 서로 다르게 나타나는 현상임니다. 예를 들자면 혈액형같은 게 있죠.

## CLNSIG 범주화
- 너무 많아서 묶을건 묶어야됨... 

In [ ]:
clinvar_df_analysis['CLNSIG'].value_counts()

In [ ]:
# # 묶기 위한 조건 설정
conditions = [
    clinvar_df_analysis['CLNSIG'].str.contains('Pathogenic|Likely_pathogenic', case=False, na=False),
    clinvar_df_analysis['CLNSIG'].str.contains('Benign|Likely_benign', case=False, na=False),
    clinvar_df_analysis['CLNSIG'].str.contains('Uncertain_significance|VUS', case=False, na=False),
    clinvar_df_analysis['CLNSIG'].str.contains('Conflicting', case=False, na=False),
    clinvar_df_analysis['CLNSIG'].str.contains('risk_factor|drug_response|association|protective|Affects', case=False, na=False)
]

# 결과 그룹명
choices = ['Pathogenic', 'Benign', 'VUS', 'Conflicting', 'Risk/Other']

# 기본값은 Unknown으로 설정
clinvar_df_analysis['CLNSIG_Group'] = np.select(conditions, choices, default='Unknown')

# 결과 확인
print(clinvar_df_analysis['CLNSIG_Group'].value_counts())

## 염색체 범주화
- 그... 다들 무슨 말인지 아시죠?

In [ ]:
def group_chrom_simple(chrom):
    # 상염색체 (1~22)
    if chrom in [str(i) for i in range(1, 23)]:
        return 'Autosome'
    # 성염색체 (X, Y)
    elif chrom in ['X', 'Y']:
        return 'Sex_Chrom'
    # 미토콘드리아
    elif chrom == 'MT':
        return 'Mitochondria'
    else:
        return 'Unknown'

clinvar_df_analysis['CHROM_Type'] = clinvar_df_analysis['CHROM'].apply(group_chrom_simple)

In [ ]:
# clinvar_df_analysis.to_csv('data/clinvar_20260208_analysis.csv', index=False)
# 태블로용으로 보냅니다 ㅅㄱㅇ

# 본게임 들어가자
- 근데 이거 염색체 시각화하려면 태블로 있어야겠는데...? 아니 염색체가 24개인데 그걸 한 화면에 어떻게 우겨넣습니까...

## CLNSIG으로 묶기

In [ ]:
clnsig_group = clinvar_df_analysis.groupby('CLNSIG_Group')['CLNSIG_Group'].count().sort_values(ascending=False)
clnsig_group

- Benign: 어... 이거는요... 그... 변이인데 뭐 누구나 하나쯤은 다 갖고있는?
- VUS: 몰?루 (진짜임)

In [ ]:
ax = sns.barplot(clnsig_group)

plt.title('CLNSIG Group에 따른 변이 수')
plt.xlabel('CLNSIG Group')
plt.yscale('log') # 이거 안하면 막대기 하나 안보임

for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, fmt='%d', padding=3, fontsize=10)

plt.tight_layout()
plt.show()

## 염색체-CLNSIG
- 상동염색체, 성염색체, 미토콘드리아(미토콘드리아도 지꺼 따로 있음)로 나눕니다.

In [ ]:
clnsig_chr_group = clinvar_df_analysis.groupby(['CHROM_Type','CLNSIG_Group']).size().unstack().fillna(0)
clnsig_chr_group

In [ ]:
ax = clnsig_chr_group.plot(kind='bar', stacked=True, ax=plt.gca(), color=sns.color_palette("Purples_r", n_colors=5))
plt.title('염색체 그룹에 따른 CLNSIG 그룹 수')
plt.xlabel('염색체')
plt.xticks(ticks=[0, 1, 2, 3], labels=['상염색체','미토콘드리아','성염색체','불명'])
plt.yscale('log') # 이거 안하면 막대기 하나 안보임
plt.tight_layout()
plt.show()

- 상염색체와 미토콘드리아는 benign 다음으로 VUS가 많은데 성염색체는 benign 다음으로 비중이 뭐 거의 또이또이 쌤쌤이다. ~~쌤쌤똔똔~~

## 염색체별 변이들 중 Pathogenic의 비율은?

In [ ]:
clnsig_chr_group = clinvar_df_analysis.groupby(['CHROM_Type','CLNSIG_Group']).size().unstack().fillna(0)
clnsig_chr_group['total'] = clnsig_chr_group.sum(axis=1)
clnsig_chr_group['Pathogenic rate'] = round(clnsig_chr_group['Pathogenic'] / clnsig_chr_group['total'] * 100, 2)
clnsig_chr_group

In [ ]:
ax = sns.barplot(clnsig_chr_group, x = 'CHROM_Type', y = 'Pathogenic rate', hue='CHROM_Type')
plt.title('염색체별 Pathogenic 비율')
plt.xlabel('염색체')
plt.ylabel('Pathogenic 비율 (%)')
plt.xticks(ticks=[0, 1, 2], labels=['상염색체','미토콘드리아','성염색체'])
plt.xlim(-0.5, 2.5)

for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10)

# 얘는 로그 빼도 됩니다. 백분율이라;;
plt.tight_layout()
plt.show()

## Pathogenic한 유전자들만 보기
- 몰?루인 VUS, 누구나 다 갖고 있는 베니건과 달리 이쪽은 터지면 큰일납니다.

In [ ]:
clnsig_pathogenic = clinvar_df_analysis.query('CLNSIG == "Pathogenic"') # Pathogenic
clnsig_pathogenic

### 어떤 돌연변이가 가장 많을까?

In [ ]:
clnsig_pathogenic_nvc = clnsig_pathogenic.groupby('CLNVC').size().sort_values(ascending=False)
clnsig_pathogenic_nvc

- 그 위에 유형 써준거 기억하시죠? single nucleotide variant는 말 그대로 나비효과입니다. 아니 염기가 하나 바뀐 것 뿐이거든요? 이게 사람 입장에서는 되에에에에ㅔ엥에에에에게 미미하잖아요? DNA 염기보다 당연히 사람이 더 크니까요. 근데 그 염기 하나가 바뀌고, 코돈이 지정하는 아미노산이 바뀌고, 단백질 접힘이 바뀌고, 접힘이 잘못된 단백질이 제 역할을 못 하는 게 사람 몸에 영향을 끼치는겁니다.
- TOP 2, TOP 3은 각각 Deletion과 Duplication이네요. 음... transposon이랑은 다릅니다. 걔는 태생이 이사 다니는 유전자예요.

In [ ]:
ax = sns.barplot(clnsig_pathogenic_nvc)
purple_color = plt.colormaps['Purples'](0.2)

# 모든 막대(patch) 가져오기
for patch in ax.patches:
    # 예: 특정 조건(height가 20 이상인 경우)의 막대만 색상 변경
    if patch.get_height() > 20000:
        pass
    else:
        patch.set_color(purple_color)

for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, padding=3, fontsize=10)

plt.title('변이 유형별 분포')
plt.ylabel('Count')
plt.show()

### 변이가 가장 많은 염색체 TOP 10
#### 염색체 종류별

In [ ]:
clnsig_pathogenic_chrom = clnsig_pathogenic.groupby('CHROM_Type').size().sort_values(ascending=False)
clnsig_pathogenic_chrom

In [ ]:
ax = sns.barplot(clnsig_pathogenic_chrom)

for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, padding=3, fontsize=10)

plt.title('염색체 종류별 분포')
plt.ylabel('Count')
plt.yscale('log')
plt.show()

- 이건 그럴수밖에 없는게 성염색체는 두개고 상염색체는 22개임...
- 미토콘드리아요? 쟤도 지꺼 있습니다.

#### 각 염색체별

In [ ]:
clnsig_pathogenic_chrom = clnsig_pathogenic.groupby('CHROM').size().sort_values(ascending=False)
clnsig_pathogenic_chrom

In [ ]:
ax = sns.barplot(clnsig_pathogenic_chrom[:10])
purple_color = plt.colormaps['Purples'](0.2)

# 모든 막대(patch) 가져오기
for patch in ax.patches:
    # 예: 특정 조건(height가 20 이상인 경우)의 막대만 색상 변경
    if patch.get_height() > 10000:
        pass
    else:
        patch.set_color(purple_color)

for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, padding=3, fontsize=10)

plt.title('변이 유형별 분포 TOP 10 (염색체별)')
plt.ylabel('Count')
plt.show()

#### 상염색체만

In [ ]:
clnsig_pathogenic_chrom_Auto = clnsig_pathogenic.query('CHROM_Type == "Autosome"').groupby('CHROM').size().sort_values(ascending=False)
clnsig_pathogenic_chrom_Auto

In [ ]:
ax = sns.barplot(clnsig_pathogenic_chrom_Auto[:10])
purple_color = plt.colormaps['Purples'](0.2)

# 모든 막대(patch) 가져오기
for patch in ax.patches:
    # 예: 특정 조건(height가 20 이상인 경우)의 막대만 색상 변경
    if patch.get_height() > 10000:
        pass
    else:
        patch.set_color(purple_color)

for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, padding=3, fontsize=10)

plt.title('변이 유형별 분포 TOP 10 (상염색체)')
plt.ylabel('Count')
plt.show()

- 17번에는 대체 뭐가 있나... 그거는 제가 태블로 대시보드로 만들어드리겠음...
- 사실 성염색체는 안해봐도 비디오인게 X가 더 큽니다 염색체가.

#### 성염색체

In [ ]:
clnsig_pathogenic_chrom_Sex = clnsig_pathogenic.query('CHROM_Type == "Sex_Chrom"').groupby('CHROM').size().sort_values(ascending=False)
clnsig_pathogenic_chrom_Sex

In [ ]:
ax = sns.barplot(clnsig_pathogenic_chrom_Sex)
purple_color = plt.colormaps['Purples'](0.2)

# 모든 막대(patch) 가져오기
for patch in ax.patches:
    # 예: 특정 조건(height가 20 이상인 경우)의 막대만 색상 변경
    if patch.get_height() > 10000:
        pass
    else:
        patch.set_color(purple_color)

for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, padding=3, fontsize=10)

plt.title('변이 유형별 분포 TOP 10 (성염색체)')
plt.ylabel('Count')
plt.yscale('log')
plt.show()

- 이게 체급차이때문에 어쩔 수 없어요. X염색체가 Y염색체에 비해서 좀 더 크고 들어있는 유전자도 더 많아서 그런겁니다. 천성적인 체급의 차이죠. X염색체는 8~900개의 유전자가 들어있지만 Y염색체에는 100개도 안 들어있어요.
- 여기서 재미있는 사실을 하나 알려드리자면... 남성은 XY 둘 다 일을 하지만 여성은 XX기 때문에 둘 중 하나만 일합니다. 그래서 생물학적으로 여성인 사람은 바소체(일 안하는 염색체)가 존재해요.
- 한가지 더 재미있는 사실. Y염색체는 재조합(염색체 두개끼리 만나서 다리 교차하고 막 그러는거)이 안 일어납니다.
- 혹시나 해서 하는 얘기지만 이건 지극히 생물학적인 팩트지 이런걸로 누가 더 우월하네 이딴걸 가리기 위함이 아님.

##### 체급차이가 있으니 비율로 봅시다. 

In [ ]:
chr_list = ['X','Y']
clnsig_pathogenic # 여기서 성염색체만 가져와서 
clnsig_pathogenic_rate = clinvar_df_analysis.query('CHROM in @chr_list').copy()
clnsig_pathogenic_rate['isPathogenic'] = clnsig_pathogenic_rate['CLNSIG'].apply(lambda x:'Pathogenic' if x == 'Pathogenic' else 'non-pathogenic')

# 그럼 비율을 봅시다... 
pathogenic_stats = clnsig_pathogenic_rate.groupby('CHROM')['isPathogenic'].value_counts(normalize=True).unstack()
# 100을 곱해서 백분율로 만들고 소수점 둘째 자리까지 반올림
pathogenic_stats_pct = (pathogenic_stats * 100).round(2)

# 결과 출력 (데이터프레임 뒤에 '%' 문자를 붙여서 시각화)
print(pathogenic_stats_pct.astype(str) + '%')

- 이거 체급이 문제가 아니고 진짜로 X염색체에 더 많은ㄷ...

In [ ]:
sns.barplot(pathogenic_stats_pct, x = 'CHROM', y = 'Pathogenic')
plt.xlabel('Chromosome')
plt.ylabel('Pathogen/Non-pathogen Percentage (%)')
plt.title('Pathogenic rate of sex chromosome')
plt.show()

### Pathogenic Gene TOP 10

In [ ]:
clnsig_pathogenic_gene = clnsig_pathogenic.groupby('GENE_SYMBOL').size().sort_values(ascending=False)
clnsig_pathogenic_gene

In [ ]:
ax = sns.barplot(clnsig_pathogenic_gene[:10])

# 모든 막대(patch) 가져오기
for patch in ax.patches:
    # 예: 특정 조건(height가 20 이상인 경우)의 막대만 색상 변경
    if patch.get_height() > 3500:
        pass
    else:
        patch.set_color(purple_color)


for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, padding=3, fontsize=10)

plt.title('변이 유형별 분포 TOP 10 (유전자별)')
plt.xlabel('유전자')
plt.ylabel('Count')
plt.show()

- BRCA는 뭔지 몰라도 안젤리나 졸리는 아시죠? 그분의 입술과 섹시미는 어유 부럽다 진짜... 근데 분석하다 말고 왜 졸리가 나옴? 안젤리나 졸리가 BRCA에 변이를 가지고 있습니다. 그래서 자기 유방을 절제하고 복원했어요. 아니 근데 왜 절제까지 하는거예요?
- 이게 그럴수밖에 없습니다. BRCA 변이는 일종의 디버프... 그러니까 상태이상에 취약해지게 만드는 디버프라고 보시면 됩니다. 저기에 변이가 있으면 유방암, 난소암에 걸릴 확률이 올라가거든요.
- 이게 유전되는거라서 본인이 BRCA 변이가 있다, 그러면 가족들 중에도 유방암이나 난소암 환자가 있을겁니다.
- 물론 저기 변이가 있다고 해서 에이씨 이번생 조졌다 이런건 아닙니다. 우리는 늘 그렇듯이 치료법을, 그리고 여러분들이 어떻게 이걸 극복할 수 있을지를 연구할테니까요.

#### Single nucleotide variant

In [ ]:
clnsig_pathogenic_snv = clnsig_pathogenic.query('CLNVC == "single_nucleotide_variant"').groupby('GENE_SYMBOL').size().sort_values(ascending=False)
clnsig_pathogenic_snv

In [ ]:
ax = sns.barplot(clnsig_pathogenic_snv[:10])

# 모든 막대(patch) 가져오기
for patch in ax.patches:
    # 예: 특정 조건(height가 20 이상인 경우)의 막대만 색상 변경
    if patch.get_height() > 1000:
        pass
    else:
        patch.set_color(purple_color)


for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, padding=3, fontsize=10)

plt.title('SNV가 가장 많은 유전자 TOP 10')
plt.xlabel('유전자')
plt.ylabel('Count')
plt.show()

- NF1은 17번 염색체에 있는 유전자인데, 세포 성장을 조절하는 종양 억제 단백질인 '뉴로파이브로민(neurofibromin)'을 생성하는 유전자입니다. 근데 이게 머고?
- 신경섬유종이 두 유형이 있는데 NF1은 1유형인 레클링하우젠 병(Recklinghausen)과 관련 있습니다. 이 병은 상염색체 우성인 병이예요.
- 2위인 FBN1은 Fibrillin-1입니다. 우리 몸에 있는 단백질 중에는 구조단백질도 있는데, 피브릴린 1은 이 미세섬유의 구성 요소 중 하나입니다.

#### Deletion

In [ ]:
clnsig_pathogenic_del = clnsig_pathogenic.query('CLNVC == "Deletion"').groupby('GENE_SYMBOL').size().sort_values(ascending=False)
clnsig_pathogenic_del

In [ ]:
ax = sns.barplot(clnsig_pathogenic_del[:10])

# 모든 막대(patch) 가져오기
for patch in ax.patches:
    # 예: 특정 조건(height가 20 이상인 경우)의 막대만 색상 변경
    if patch.get_height() > 1000:
        pass
    else:
        patch.set_color(purple_color)


for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, padding=3, fontsize=10)

plt.title('Deletion이 가장 많은 유전자 TOP 10')
plt.xlabel('유전자')
plt.ylabel('Count')
plt.show()

#### Duplication

In [ ]:
clnsig_pathogenic_du = clnsig_pathogenic.query('CLNVC == "Duplication"').groupby('GENE_SYMBOL').size().sort_values(ascending=False)
clnsig_pathogenic_du

In [ ]:
ax = sns.barplot(clnsig_pathogenic_du[:10])

# 모든 막대(patch) 가져오기
for patch in ax.patches:
    # 예: 특정 조건(height가 20 이상인 경우)의 막대만 색상 변경
    if patch.get_height() > 500:
        pass
    else:
        patch.set_color(purple_color)


for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, padding=3, fontsize=10)

plt.title('Duplication이 가장 많은 유전자 TOP 10')
plt.xlabel('유전자')
plt.ylabel('Count')
plt.show()

### Insertion이 제일 많은 유전자

In [ ]:
clnsig_pathogenic_in = clnsig_pathogenic.query('CLNVC == "Insertion"').groupby('GENE_SYMBOL').size().sort_values(ascending=False)
clnsig_pathogenic_in

In [ ]:
ax = sns.barplot(clnsig_pathogenic_in[:10])

# 모든 막대(patch) 가져오기
for patch in ax.patches:
    # 예: 특정 조건(height가 20 이상인 경우)의 막대만 색상 변경
    if patch.get_height() > 100:
        pass
    else:
        patch.set_color(purple_color)


for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, padding=3, fontsize=10)

plt.title('Insertion이 가장 많은 유전자 TOP 10')
plt.xlabel('유전자')
plt.ylabel('Count')
plt.show()

### Inversion이 가장 많은 유전자

In [ ]:
clnsig_pathogenic_inv = clnsig_pathogenic.query('CLNVC == "Inversion"').groupby('GENE_SYMBOL').size().sort_values(ascending=False)
clnsig_pathogenic_inv

In [ ]:
ax = sns.barplot(clnsig_pathogenic_inv[:10])

# 모든 막대(patch) 가져오기
for patch in ax.patches:
    # 예: 특정 조건(height가 20 이상인 경우)의 막대만 색상 변경
    if patch.get_height() > 1:
        pass
    else:
        patch.set_color(purple_color)


for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, padding=3, fontsize=10)

plt.title('Inversion이 가장 많은 유전자 TOP 10')
plt.xlabel('유전자')
plt.ylabel('Count')
plt.show()

### Indel이 가장 많은 유전자

In [ ]:
clnsig_pathogenic_indel = clnsig_pathogenic.query('CLNVC == "Indel"').groupby('GENE_SYMBOL').size().sort_values(ascending=False)
clnsig_pathogenic_indel

In [ ]:
ax = sns.barplot(clnsig_pathogenic_indel[:10])

# 모든 막대(patch) 가져오기
for patch in ax.patches:
    # 예: 특정 조건(height가 20 이상인 경우)의 막대만 색상 변경
    if patch.get_height() > 100:
        pass
    else:
        patch.set_color(purple_color)


for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, padding=3, fontsize=10)

plt.title('Inversion이 가장 많은 유전자 TOP 10')
plt.xlabel('유전자')
plt.ylabel('Count')
plt.show()